# 다대포 해수욕장 수질 오염 예측 프로젝트 (Spatio-Temporal & CSO Modeling)

이 노트북은 전체 데이터 전처리부터 XGBoost 모델 훈련까지의 파이프라인을 포함하고 있습니다.

## Step 1: 수질 원본 데이터 공간 확장 전처리

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

def clean_microbial_value(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    if not val_str:
        return np.nan
    if '<' in val_str:
        try:
            return float(val_str.replace('<', '')) * 0.5
        except:
            return 0.5
    if '>' in val_str:
        try:
            return float(val_str.replace('>', ''))
        except:
            return np.nan
    val_str = val_str.replace(',', '')
    try:
        return float(val_str)
    except:
        return np.nan

def get_distance(row):
    detail = str(row['examinLcDetail']).upper()
    beach = str(row['beachKoreanNm']).upper()
    
    # 다대포 서측(강이랑 가까운 해변)은 기본 거리 0.0km 시작
    if '서측' in beach:
        base_dist = 0.0
    # 다대포 동측(강이랑 먼 해변)은 기본 거리 0.5km 시작
    else: 
        base_dist = 0.5
        
    # 각 해변 내에서의 세부 지점(A,B,C) 더하기
    if 'A' in detail or '우측' in detail:
        return base_dist + 0.0
    elif 'B' in detail or '중앙' in detail:
        return base_dist + 0.25
    elif 'C' in detail or '좌측' in detail:
        return base_dist + 0.5
        
    return base_dist + 0.25 # Default to center if unknown

def preprocess_water_quality():
    # Use glob to avoid encoding issues with korean filenames in different environments
    files = glob.glob("C:\\Sandbox\\Water_Quality\\*다대포*.csv")
    
    df_list = []
    for f in files:
        try:
            df = pd.read_csv(f, encoding='utf-8-sig')
        except:
            df = pd.read_csv(f, encoding='cp949')
        df_list.append(df)
            
    if not df_list:
        print("No Dadaepo water quality files found.")
        return
        
    df_all = pd.concat(df_list, ignore_index=True)
    
    # Drop rows without date
    df_clean = df_all.dropna(subset=['examinDe']).copy()
    df_clean = df_clean[df_clean['examinDe'].str.strip() != '']
    df_clean['examinDe'] = pd.to_datetime(df_clean['examinDe'], errors='coerce')
    df_clean = df_clean.dropna(subset=['examinDe'])
    
    # Clean microbial values
    df_clean['ecoli_max'] = df_clean['coliDetectCn'].apply(clean_microbial_value)
    df_clean['enterococcus_max'] = df_clean['entrcccsDetectCn'].apply(clean_microbial_value)
    
    # Exceedance Labels
    df_clean['ecoli_exceed'] = df_clean['ecoli_max'] > 500
    df_clean['enterococcus_exceed'] = df_clean['enterococcus_max'] > 100
    df_clean['any_exceed'] = df_clean['ecoli_exceed'] | df_clean['enterococcus_exceed']
    
    # Add Spatial Feature
    df_clean['distance_from_estuary_km'] = df_clean.apply(get_distance, axis=1)
    
    # Keep essential columns for modeling
    cols_to_keep = ['examinDe', 'beachKoreanNm', 'examinLcDetail', 'distance_from_estuary_km', 
                    'ecoli_max', 'enterococcus_max', 'any_exceed']
    
    # We DO NOT groupby here anymore! We keep all 305 sample rows.
    df_samples = df_clean[cols_to_keep].copy()
    
    print(f"Total Exceedances (Sample level): {df_samples['any_exceed'].sum()} out of {len(df_samples)} samples")
    
    # Save
    out_dir = "C:\\Sandbox\\Preprocessed"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "dadaepo_water_samples.csv")
    df_samples.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Saved {len(df_samples)} spatial samples to {out_path}")

if __name__ == "__main__":
    preprocess_water_quality()


In [ ]:
preprocess_water_quality()

## Step 2: 강변하수처리장 일일방류량 전처리

In [ ]:
import pandas as pd
import numpy as np
import os

def preprocess_sewage():
    file_path = "C:\\Sandbox\\Discharge\\강변사업단 일일방류량.csv"
    
    if not os.path.exists(file_path):
        print(f"Sewage data not found at {file_path}")
        return
        
    # Read sewage data
    df = pd.read_csv(file_path, encoding='cp949')
    
    # Rename columns
    df.columns = ['date_str', 'sewage_volume', 'ph', 'bod', 'cod', 'ss', 'tn', 'tp']
    
    # Convert date
    df['date_str'] = df['date_str'].astype(str)
    df['date'] = pd.to_datetime(df['date_str'], format='%Y%m%d')
    df = df.sort_values('date').reset_index(drop=True)
    
    # We will let merge_datasets.py handle the lag and CSO Flag computation
    # because it has the weather data already processed.
    
    cols_to_keep = ['date', 'sewage_volume']
    
    out_dir = "C:\\Sandbox\\Preprocessed"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "sewage_daily.csv")
    df[cols_to_keep].to_csv(out_path, index=False)
    print(f"Saved sewage data to {out_path}")

if __name__ == "__main__":
    preprocess_sewage()


In [ ]:
preprocess_sewage()

## Step 3: 날씨/센서/하수 병합 및 CSO_Flag 파생 변수 생성

In [ ]:
import pandas as pd
import numpy as np
import os

def build_master_dataset():
    water_path = "C:\\Sandbox\\Preprocessed\\dadaepo_water_samples.csv"
    weather_path = "C:\\Sandbox\\Weather\\dadaepo_weather_2014_2026.csv"
    sensor_path = "C:\\Sandbox\\Preprocessed\\nakdong_sensor_daily.csv"
    visitor_path = "C:\\Sandbox\\Preprocessed\\dadaepo_visitors_daily.csv"
    discharge_path = "C:\\Sandbox\\Preprocessed\\dadaepo_discharge_daily.csv"
    sewage_path = "C:\\Sandbox\\Preprocessed\\sewage_daily.csv"
    tide_path = "C:\\Sandbox\\Preprocessed\\dadaepo_tide_daily.csv"
    
    print("Loading datasets...")
    df_water = pd.read_csv(water_path)
    df_weather = pd.read_csv(weather_path)
    df_sensor = pd.read_csv(sensor_path)
    
    # Load Visitors
    has_visitors = False
    if os.path.exists(visitor_path):
        df_visitor = pd.read_csv(visitor_path)
        has_visitors = True
    else:
        df_visitor = pd.DataFrame(columns=['date', 'visitor_count'])
        
    # Load Discharge (Nakdong River)
    has_discharge = False
    if os.path.exists(discharge_path):
        df_discharge = pd.read_csv(discharge_path)
        has_discharge = True
    else:
        df_discharge = pd.DataFrame(columns=['date', 'discharge_volume'])
        
    # Load Tide
    has_tide = False
    if os.path.exists(tide_path):
        df_tide = pd.read_csv(tide_path)
        has_tide = True
    else:
        df_tide = pd.DataFrame(columns=['date', 'tide_max', 'tide_min', 'tide_range'])
        
    # Load Sewage
    has_sewage = False
    if os.path.exists(sewage_path):
        df_sewage = pd.read_csv(sewage_path)
        has_sewage = True
    else:
        df_sewage = pd.DataFrame(columns=['date', 'sewage_volume'])
        
    df_water['date_obj'] = pd.to_datetime(df_water['examinDe'])
    df_sensor['date_obj'] = pd.to_datetime(df_sensor['date'])
    df_visitor['date_obj'] = pd.to_datetime(df_visitor['date'])
    df_discharge['date_obj'] = pd.to_datetime(df_discharge['date'])
    df_tide['date_obj'] = pd.to_datetime(df_tide['date'])
    df_sewage['date_obj'] = pd.to_datetime(df_sewage['date'])
    
    # Drop string 'date' column so it doesn't cause merge conflicts
    for d in [df_discharge, df_tide, df_sewage]:
        if 'date' in d.columns:
            d.drop(columns=['date'], inplace=True)
    
    # Process Weather Data
    df_weather['time'] = pd.to_datetime(df_weather['time'])
    df_weather['date'] = df_weather['time'].dt.date
    daily_weather = df_weather.groupby('date').agg(
        precip_daily=('precipitation_mm', 'sum'),
        temp_daily_mean=('temperature_2m', 'mean'),
        wind_speed_daily_max=('wind_speed_m_s', 'max')
    ).reset_index()
    daily_weather['date_obj'] = pd.to_datetime(daily_weather['date'])
    
    # Merge all daily features into one big Feature Table
    feature_table = pd.merge(daily_weather, df_sensor, on='date_obj', how='outer')
    feature_table = pd.merge(feature_table, df_visitor, on='date_obj', how='outer')
    feature_table = pd.merge(feature_table, df_discharge, on='date_obj', how='outer')
    feature_table = pd.merge(feature_table, df_tide, on='date_obj', how='outer')
    feature_table = pd.merge(feature_table, df_sewage, on='date_obj', how='outer')
    
    feature_table = feature_table.sort_values('date_obj').reset_index(drop=True)
    
    # Create Lag features
    feature_table['precip_1d_lag'] = feature_table['precip_daily'].shift(1)
    feature_table['precip_2d_sum_lag'] = feature_table['precip_daily'].rolling(2).sum().shift(1)
    feature_table['precip_3d_sum_lag'] = feature_table['precip_daily'].rolling(3).sum().shift(1)
    
    feature_table['discharge_1d_lag'] = feature_table['discharge_volume'].shift(1)
    feature_table['discharge_3d_sum_lag'] = feature_table['discharge_volume'].rolling(3).sum().shift(1)
    
    feature_table['temp_1d_lag'] = feature_table['temp_daily_mean'].shift(1)
    feature_table['wind_max_1d_lag'] = feature_table['wind_speed_daily_max'].shift(1)
    
    feature_table['sewage_discharge_1d_lag'] = feature_table['sewage_volume'].shift(1)
    feature_table['sewage_discharge_3d_sum_lag'] = feature_table['sewage_volume'].rolling(3).sum().shift(1)
    
    # Calculate CSO Flag
    # CSO condition: Sewage Volume on the day prior to testing >= 450,000 AND 3-day accumulated rain >= 5.0mm
    cso_cond = (feature_table['sewage_discharge_1d_lag'] >= 450000) & (feature_table['precip_3d_sum_lag'] >= 5.0)
    feature_table['CSO_Flag'] = cso_cond.astype(int)
    
    # Sensor and visitor lag features (only 1 day prior)
    lag_cols = [c for c in df_sensor.columns if c not in ['date', 'date_obj']] + ['visitor_count']
    for col in lag_cols:
        if col in feature_table.columns:
            feature_table[f"{col}_1d_lag"] = feature_table[col].shift(1)
            
    # Merge onto water quality target table (dadaepo_water_samples.csv has 305 rows)
    master_df = pd.merge(df_water, feature_table, on='date_obj', how='left')
    
    # Clean up redundant columns
    cols_to_drop = ['date_y', 'date_obj', 'date_x'] + [c for c in feature_table.columns if not c.endswith('_lag') and c not in ['date', 'tide_max', 'tide_min', 'tide_range', 'CSO_Flag']]
    
    # Rename date_x back to date
    master_df = master_df.rename(columns={'date_x': 'date'})
    master_df = master_df.drop(columns=[c for c in cols_to_drop if c in master_df.columns], errors='ignore')
    
    out_dir = "C:\\Sandbox\\Preprocessed"
    out_path = os.path.join(out_dir, "master_dataset_v2.csv")
    master_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    
    print(f"Master dataset V2 created with {len(master_df)} rows and {len(master_df.columns)} features.")
    print(f"Total CSO events observed in test samples: {master_df['CSO_Flag'].sum()}")
    print(f"Saved to {out_path}")
    
if __name__ == "__main__":
    build_master_dataset()


In [ ]:
build_master_dataset()

## Step 4: XGBoost V2 훈련 및 결과 시각화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
import os

# Set font for Korean text in plots
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def run_regression_modeling():
    data_path = "C:\\Sandbox\\Preprocessed\\master_dataset_v2.csv"
    out_dir = "C:\\Sandbox\\Preprocessed"
    
    df = pd.read_csv(data_path)
    
    # Calculate Risk Index
    df['ecoli_risk'] = df['ecoli_max'] / 500.0
    df['entero_risk'] = df['enterococcus_max'] / 100.0
    df['risk_index'] = df[['ecoli_risk', 'entero_risk']].max(axis=1)
    
    # Predict log(1+x) to handle scale
    df['log_ecoli'] = np.log1p(df['ecoli_max'])
    df['log_entero'] = np.log1p(df['enterococcus_max'])
    
    # Base Features (Original 90-row model baseline features for fair comparison, minus discharge as we use sewage now)
    features = ['precip_1d_lag', 'precip_2d_sum_lag', 'precip_3d_sum_lag', 'temp_1d_lag', 'wind_max_1d_lag', 'discharge_1d_lag', 'discharge_3d_sum_lag']
    
    # V2 Advanced Features (Spatial + Sewage CSO)
    # We include all base features, plus spatial (distance), sensors, visitors, tide, and sewage.
    adv_features = features + [
        'sensor_turbidity_max_1d_lag', 'sensor_salinity_min_1d_lag', 'sensor_temp_mean_1d_lag', 
        'visitor_count_1d_lag', 
        'distance_from_estuary_km', 'sewage_discharge_1d_lag', 'sewage_discharge_3d_sum_lag', 'CSO_Flag'
    ]
    
    X_base = df[features]
    X_adv = df[adv_features]
    y_reg = df[['log_ecoli', 'log_entero']]
    y_bin = df['any_exceed'].astype(int)
    
    # MICE Imputer
    imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=42), random_state=42, max_iter=10)
    
    # Ridge Pipeline
    ridge_pipe = Pipeline([
        ('imputer', imputer),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0, random_state=42))
    ])
    
    # XGBoost Pipeline
    # Data is now 305 rows, we can afford slightly deeper trees, but still keep it robust.
    xgb_pipe = Pipeline([
        ('imputer', imputer),
        ('model', XGBRegressor(
            n_estimators=100, 
            learning_rate=0.05, 
            max_depth=4, 
            subsample=0.8, 
            colsample_bytree=0.8, 
            random_state=42
        ))
    ])
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    
    def evaluate_model(pipe, X, y_reg, y_bin):
        rmses = []
        r2s = []
        aucs = []
        
        preds_all = np.zeros((len(y_reg), 2))
        
        for train_idx, test_idx in cv.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train_reg, y_test_reg = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
            y_test_bin = y_bin.iloc[test_idx]
            
            pipe.fit(X_train, y_train_reg)
            preds = pipe.predict(X_test)
            preds_all[test_idx] = preds
            
            rmses.append(np.sqrt(mean_squared_error(y_test_reg, preds)))
            r2s.append(r2_score(y_test_reg, preds))
            
            # Reconstruct binary prediction for AUC
            pred_ecoli = np.expm1(preds[:, 0])
            pred_entero = np.expm1(preds[:, 1])
            pred_risk = np.maximum(pred_ecoli / 500.0, pred_entero / 100.0)
            
            try:
                aucs.append(roc_auc_score(y_test_bin, pred_risk))
            except ValueError:
                pass
                
        return np.mean(rmses), np.mean(r2s), np.mean(aucs), preds_all

    # Evaluate Ridge
    ridge_base_rmse, ridge_base_r2, ridge_base_auc, ridge_base_preds = evaluate_model(ridge_pipe, X_base, y_reg, y_bin)
    ridge_adv_rmse, ridge_adv_r2, ridge_adv_auc, ridge_adv_preds = evaluate_model(ridge_pipe, X_adv, y_reg, y_bin)
    
    # Evaluate XGBoost
    xgb_base_rmse, xgb_base_r2, xgb_base_auc, xgb_base_preds = evaluate_model(xgb_pipe, X_base, y_reg, y_bin)
    xgb_adv_rmse, xgb_adv_r2, xgb_adv_auc, xgb_adv_preds = evaluate_model(xgb_pipe, X_adv, y_reg, y_bin)
    
    # Save Results
    with open(os.path.join(out_dir, "regression_results.txt"), 'w', encoding='utf-8') as f:
        f.write("=== XGBoost V2 (공간 확장 + 하수처리장 CSO 결합) 결과 ===\n\n")
        f.write("1. Ridge (Baseline):\n")
        f.write(f"   RMSE: {ridge_base_rmse:.3f} | R^2: {ridge_base_r2:.3f} | ROC-AUC: {ridge_base_auc:.3f}\n\n")
        
        f.write("2. Ridge (Advanced V2):\n")
        f.write(f"   RMSE: {ridge_adv_rmse:.3f} | R^2: {ridge_adv_r2:.3f} | ROC-AUC: {ridge_adv_auc:.3f}\n\n")
        
        f.write("3. XGBoost (Baseline):\n")
        f.write(f"   RMSE: {xgb_base_rmse:.3f} | R^2: {xgb_base_r2:.3f} | ROC-AUC: {xgb_base_auc:.3f}\n\n")
        
        f.write("4. XGBoost (Advanced V2):\n")
        f.write(f"   RMSE: {xgb_adv_rmse:.3f} | R^2: {xgb_adv_r2:.3f} | ROC-AUC: {xgb_adv_auc:.3f}\n\n")
        
    # Scatter plot Actual vs Predicted (using Advanced XGBoost V2)
    plt.figure(figsize=(10, 5))
    
    actual_ecoli = np.expm1(y_reg['log_ecoli'])
    actual_entero = np.expm1(y_reg['log_entero'])
    
    pred_ecoli = np.expm1(xgb_adv_preds[:, 0])
    pred_entero = np.expm1(xgb_adv_preds[:, 1])
    
    plt.subplot(1, 2, 1)
    plt.scatter(actual_ecoli, pred_ecoli, alpha=0.6, edgecolors='k')
    plt.plot([0, max(actual_ecoli)], [0, max(actual_ecoli)], 'r--')
    plt.xscale('symlog')
    plt.yscale('symlog')
    plt.xlabel('Actual E.coli')
    plt.ylabel('Predicted E.coli')
    plt.title('XGBoost V2 E.coli Prediction')
    
    plt.subplot(1, 2, 2)
    plt.scatter(actual_entero, pred_entero, alpha=0.6, edgecolors='k')
    plt.plot([0, max(actual_entero)], [0, max(actual_entero)], 'r--')
    plt.xscale('symlog')
    plt.yscale('symlog')
    plt.xlabel('Actual Enterococcus')
    plt.ylabel('Predicted Enterococcus')
    plt.title('XGBoost V2 Enterococcus Prediction')
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "regression_scatter.png"))

    # Feature Importance for XGBoost V2
    xgb_pipe.fit(X_adv, y_reg)
    
    avg_importance = xgb_pipe.named_steps['model'].feature_importances_
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x=avg_importance, y=X_adv.columns)
    plt.title('XGBoost V2 Feature Importances (Spatial + CSO)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "feature_importance_xgboost.png"))

if __name__ == "__main__":
    run_regression_modeling()


In [ ]:
run_regression_modeling()